# Week 07 — From reusable functions to a Streamlit application

**Seismic Python & AI Engineering | Hands-on notebook**

A professional application separates computation from the user interface. We will review and improve the small Streamlit example included in `apps/`.

**Estimated independent study:** 4–5 hours in this notebook, plus 2–3 hours on repository exercises and review.  
**Data:** entirely synthetic; no external API, proprietary dataset, or GPU required.  
**Prerequisite:** complete the root README setup and open Jupyter from the repository root.

### Learning outcomes
- Explain model/view separation and why UI code must not own numerical logic.
- Build JSON-safe payloads and avoid leaking internal NumPy types.
- Validate interactive parameters and display units clearly.
- Identify practical application tests and deployment constraints.

### How to study
Read the explanations, execute the worked cells in order, complete the **Your turn** tasks, and implement the week's function in `src/seismic_toolkit/challenges.py`. Run the targeted exercise test to assess progress. The initial baseline tests should pass even before you complete the challenges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from seismic_toolkit import __version__
print(f"Seismic toolkit version: {__version__}")
np.set_printoptions(precision=4, suppress=True)

## 1. Application structure
`apps/streamlit_app.py` is an optional interface; the numerical work lives in `seismic_toolkit.pipeline`, `synthetic` and `attributes`. Keep your functions usable from a notebook, a CLI and a UI without importing Streamlit into the math modules.

Run the app outside Jupyter with:

```bash
python -m pip install -e '.[app]'
streamlit run apps/streamlit_app.py
```

The browser interface should state that its synthetic output is educational, not suitable for reservoir decisions.

In [ ]:
from seismic_toolkit.pipeline import run_synthetic_demo
result = run_synthetic_demo(dt=0.001, delay_samples=6)
print("True delay (ms):", result["true_delay_s"] * 1e3)
print("Estimated delay (ms):", result["estimated_delay_s"] * 1e3)

## 2. Why serialization boundaries matter
Web applications transmit JSON-like data. NumPy arrays and scalar values are not always JSON-serializable in their native form. Convert to Python lists or scalars at the **boundary**, without contaminating your numerical core with interface concerns.

In [ ]:
import json
x = np.array([0., 0.5, -1., 0.25])
dt_s = 0.002
payload = {"time_s": (np.arange(x.size) * dt_s).tolist(), "amplitude": x.tolist()}
print(json.dumps(payload, indent=2))
assert json.loads(json.dumps(payload))["time_s"][2] == 0.004

## 3. UI state and validation
Streamlit widgets can rerun the script whenever a value changes. Do not rely on hidden mutable globals for scientific calculations. Validate numerical ranges in the underlying library as well as at the widget layer; an API or CLI may call the same functions without a UI.

**Your turn A:** inspect `apps/streamlit_app.py`. Find the widget inputs, the reusable computation function, the physical units displayed and the disclaimer. Explain where you would insert caching, and whether this simple operation actually needs it.

In [ ]:
from pathlib import Path
paths = [Path("apps/streamlit_app.py"), Path("../apps/streamlit_app.py")]
app_file = next((p for p in paths if p.exists()), None)
print("App file:", app_file)
if app_file is not None:
    text = app_file.read_text(encoding="utf-8")
    print("UI imports toolkit:", "from seismic_toolkit.pipeline import run_synthetic_demo" in text)

## 4. Independent implementation — Week 7 challenge
Implement `prepare_plot_payload(trace, dt)` in `challenges.py`.

Return a JSON-serializable dictionary with exactly the keys `time_s` and `amplitude`, each holding an ordinary Python list of floats. Start the time axis at zero and increment it by `dt` in seconds. Reject invalid `dt`, empty arrays and nonfinite samples.

**Your turn B:** create a test asserting `json.dumps(payload)` succeeds and that the time axis matches the trace length. Then build a notebook plot from the payload.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(payload["time_s"], payload["amplitude"], marker="o")
ax.set(xlabel="Time (s)", ylabel="Amplitude", title="Plot from a JSON-safe payload")
ax.grid(alpha=0.25)
plt.show()

## 5. Practical UI acceptance tests
A usable scientific interface should: indicate units; reject impossible values; preserve deterministic behavior; display algorithm version and limitations; not reveal private information; and make it easy to reproduce a result from saved parameters.

**Applied problem:** add a switch to view either baseline alone or baseline + monitor. Keep all numerical calculations in the package. Write a manual test checklist for changes to sampling interval, time delay and plot labels.

**Stretch:** add a CSV **download of synthetic data only** and ensure any file path is explicitly controlled.

---

## Weekly submission and self-review

1. Complete the independent exercise in `src/seismic_toolkit/challenges.py`.
2. Run: `python -m pytest -o addopts='' -m exercise -k week7 -q`.
3. Run baseline regression tests: `python -m pytest -q`.
4. Commit a focused change to a feature branch; document what changed and why.
5. Record one failure mode, one design decision, and one next-step question in your learning journal.

**Further work:** see [`../exercises/week07.md`](../exercises/week07.md) for a graded problem, extra practice and acceptance criteria. Compare with `solutions/reference_challenges.py` only **after** attempting your own solution.

**Important:** all datasets and algorithms are educational demonstrations, not validated reservoir-interpretation systems.